# Evaluate accuracy from the retained latest checkpoints

Use this notebook when old prediction artifacts have already been deleted but the latest Drive checkpoints still exist.

It does not train any model. It reruns only inference for:

- `pre_hidden_1_16_r16_run_02`
- `reverse_bottleneck_fusion_s0_r4_run_01`

The notebook exports Dice, HD95, Jaccard/IoU, voxel accuracy, foreground voxel accuracy, mean foreground accuracy, and pancreas accuracy. A missing checkpoint is reported and skipped instead of stopping the full batch.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/ihatesea69/MSCAF-TransUNet.git'
REPO_BRANCH = 'main'
PROJECT_DIR = Path('/content/MSCAF-TransUNet')
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_EXPORT_DIR = DRIVE_ROOT / 'transunet_colab_outputs'
OUTPUT_DIR = DRIVE_EXPORT_DIR / 'accuracy_exports'


In [ ]:
import shutil
import subprocess
import sys

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')],
    check=True,
)


In [ ]:
COMMON_DATASET_LOCATIONS = [
    DRIVE_ROOT / 'datasets' / 'Synapse' / 'test_vol_h5',
    DRIVE_ROOT / 'Synapse' / 'test_vol_h5',
    DRIVE_ROOT / 'TransUNet-Medical-Image-Segmentation' / 'data' / 'Synapse' / 'test_vol_h5',
]

TEST_DATA_DIR = next((path for path in COMMON_DATASET_LOCATIONS if path.exists()), None)
if TEST_DATA_DIR is None:
    discovered = [path for path in DRIVE_ROOT.rglob('test_vol_h5') if path.is_dir()]
    if not discovered:
        raise FileNotFoundError('Synapse test_vol_h5 was not found on Google Drive.')
    TEST_DATA_DIR = discovered[0]

EXPERIMENTS = [
    {
        'run_id': 'pre_hidden_1_16_r16_run_02',
        'snapshot_name': 'TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224_attn-pre_hidden-1_16-r16',
        'extra_args': [],
    },
    {
        'run_id': 'reverse_bottleneck_fusion_s0_r4_run_01',
        'snapshot_name': 'TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224_attn-pre_hidden-1_16-r16_ra-ra_fusion-s0-r4',
        'extra_args': ['--ra_mode', 'ra_fusion', '--ra_scales', '0', '--ra_reduction', '4'],
    },
]

print('Synapse test data:', TEST_DATA_DIR)
for experiment in EXPERIMENTS:
    checkpoint = DRIVE_EXPORT_DIR / 'resume_checkpoints' / experiment['snapshot_name'] / 'latest_checkpoint.pth'
    print(experiment['run_id'], 'checkpoint exists:', checkpoint.exists(), checkpoint)


In [ ]:
import json
import os
import re
import subprocess
import sys

DICE_PATTERN = re.compile(r'Testing performance in best val model: mean_dice : ([0-9.]+) mean_hd95 : ([0-9.]+)(?: mean_jaccard : ([0-9.]+))?')
ACCURACY_PATTERN = re.compile(
    r'Accuracy performance: voxel_accuracy : ([0-9.]+) '
    r'foreground_voxel_accuracy : ([0-9.]+) '
    r'mean_foreground_accuracy : ([0-9.]+) '
    r'pancreas_accuracy : ([0-9.]+)'
)

def run_stream(command, env):
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='')
        lines.append(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return ''.join(lines)

def percent(value):
    return f'{100 * value:.4f}%'

results = []
for experiment in EXPERIMENTS:
    checkpoint_dir = DRIVE_EXPORT_DIR / 'resume_checkpoints' / experiment['snapshot_name']
    checkpoint = checkpoint_dir / 'latest_checkpoint.pth'
    if not checkpoint.exists():
        results.append({
            'run_id': experiment['run_id'],
            'status': 'missing_checkpoint',
            'checkpoint': str(checkpoint),
        })
        continue

    command = [
        sys.executable, '-u', 'test.py',
        '--dataset', 'Synapse',
        '--vit_name', 'R50-ViT-B_16',
        '--img_size', '224',
        '--num_classes', '9',
        '--n_skip', '3',
        '--vit_patches_size', '16',
        '--max_iterations', '30000',
        '--max_epochs', '150',
        '--batch_size', '24',
        '--base_lr', '0.01',
        '--seed', '1234',
        '--deterministic', '1',
        '--attention_mode', 'pre_hidden',
        '--attention_scales', '1/16',
        '--attention_reduction', '16',
        *experiment['extra_args'],
    ]
    env = os.environ.copy()
    env['TRANSUNET_TEST_DATA_DIR'] = str(TEST_DATA_DIR)
    env['TRANSUNET_CHECKPOINT_DIR'] = str(checkpoint_dir)

    print('\n===', experiment['run_id'], '===')
    output = run_stream(command, env)
    dice_match = DICE_PATTERN.search(output)
    accuracy_match = ACCURACY_PATTERN.search(output)
    if not dice_match or not accuracy_match:
        raise RuntimeError(f'Expected metrics were not found for {experiment["run_id"]}.')

    metrics = {
        'mean_dice': float(dice_match.group(1)),
        'mean_hd95': float(dice_match.group(2)),
        'mean_jaccard': float(dice_match.group(3)) if dice_match.group(3) else None,
        'voxel_accuracy': float(accuracy_match.group(1)),
        'foreground_voxel_accuracy': float(accuracy_match.group(2)),
        'mean_foreground_accuracy': float(accuracy_match.group(3)),
        'pancreas_accuracy': float(accuracy_match.group(4)),
    }
    results.append({
        'run_id': experiment['run_id'],
        'status': 'completed',
        'checkpoint': str(checkpoint),
        'metrics': metrics,
    })

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
json_path = OUTPUT_DIR / 'latest_checkpoint_accuracy_results.json'
markdown_path = OUTPUT_DIR / 'latest_checkpoint_accuracy_summary.md'
json_path.write_text(json.dumps({'results': results}, indent=2), encoding='utf-8')

lines = [
    '# Latest Checkpoint Accuracy Results',
    '',
    '| Run | Status | Mean DSC | Mean HD95 | Mean Jaccard | Voxel accuracy | Foreground voxel accuracy | Mean foreground accuracy | Pancreas accuracy |',
    '|---|---|---:|---:|---:|---:|---:|---:|---:|',
]
for result in results:
    metrics = result.get('metrics')
    if metrics:
        mean_jaccard = metrics.get('mean_jaccard')
        mean_jaccard_text = f'{mean_jaccard:.6f}' if mean_jaccard is not None else '-'
        lines.append(
            f'| `{result["run_id"]}` | completed | {metrics["mean_dice"]:.6f} | {metrics["mean_hd95"]:.6f} | '
            f'{mean_jaccard_text} | {percent(metrics["voxel_accuracy"])} | {percent(metrics["foreground_voxel_accuracy"])} | '
            f'{percent(metrics["mean_foreground_accuracy"])} | {percent(metrics["pancreas_accuracy"])} |'
        )
    else:
        lines.append(f'| `{result["run_id"]}` | {result["status"]} | - | - | - | - | - | - | - |')
markdown_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

print('\n' + markdown_path.read_text(encoding='utf-8'))
print('JSON:', json_path)
print('Markdown:', markdown_path)


## Next step

Download or share `latest_checkpoint_accuracy_results.json` after the evaluation finishes. The report can then be updated with the measured accuracy values. If a checkpoint is missing, that run cannot be reconstructed without rerunning training.
